# Gestionnaire de tâches avec Tkinter

Ce TP construit une application de bureau complète, un gestionnaire de tâches : une saisie et un bouton pour ajouter une tâche, la liste des tâches, des boutons pour les marquer faites ou les supprimer, et un filtre (toutes, à faire, faites). Elle suit l'organisation Modèle-Vue-Contrôleur du cours, en trois fichiers que les cellules écrivent : le modèle dans `modele.py`, la vue dans `vue.py`, le contrôleur et le lancement dans `app.py`. L'application tourne hors du notebook, dans sa propre fenêtre.

Il faut un écran pour l'afficher :

- **sur votre machine**, dans Jupyter ou VS Code, avec un Python qui a Tk : `python -m tkinter` doit ouvrir une petite fenêtre (sous Linux, installez sinon `python3-tk` avec le gestionnaire de paquets). Depuis Colab, *Fichier › Télécharger › Télécharger le fichier .ipynb* récupère ce notebook ;
- **dans le Codespace**, les fenêtres s'ouvrent sur son bureau : dans l'onglet *Ports*, à côté du terminal, survolez la ligne du port 6080, *Bureau / Desktop*, et cliquez sur l'icône du globe, *Open in Browser*, puis sur *Connecter* dans l'onglet qui s'ouvre, sans mot de passe ;
- **dans Colab**, il n'y a pas d'écran : le modèle s'écrit et se teste, mais les fenêtres ne s'ouvrent pas. Le TP jumeau, la même application en ipywidgets, y tourne entièrement.

C'est un TP linéaire : chaque partie s'appuie sur la précédente. Si vous bloquez, exécutez la solution et passez à la suite.

## Lancer une fenêtre depuis le notebook

La fonction `lancer` exécute Python dans un processus à part, comme `python app.py` dans un terminal : la cellule reste en cours d'exécution tant que la fenêtre est ouverte, puis affiche ce que le programme a écrit (ses `print`, ou l'erreur qui l'a arrêté). Sans écran, elle le dit et ne lance rien.

La dernière ligne ouvre la fenêtre de test de Tk : fermez-la pour rendre la main au notebook.

In [ ]:
import subprocess
import sys


def lancer(*arguments: str) -> None:
  ecran = subprocess.run(
    [sys.executable, "-c", "import tkinter; tkinter.Tk().destroy()"],
    capture_output=True,
    text=True,
  )
  if ecran.returncode != 0:
    cause = ecran.stderr.strip().splitlines()[-1]
    print(f"Pas d'écran ici, rien n'est lancé ({cause}).")
    return
  resultat = subprocess.run(
    [sys.executable, *arguments], capture_output=True, text=True
  )
  print(resultat.stdout, resultat.stderr, sep="")


lancer("-m", "tkinter")

## 1. Le modèle

Écrivez `modele.py`, du Python pur, sans interface : c'est lui qui porte les règles de l'application.

- `FILTRES = ("toutes", "à faire", "faites")`, les filtres que l'interface proposera.
- `Tache`, une dataclass : un `titre` (`str`) et `faite` (`bool`, `False` par défaut). Le titre perd ses espaces de début et de fin, et un titre vide lève une `ValueError`.
- `ListeTaches`, une dataclass dont l'unique attribut, `taches`, est une liste de `Tache`, vide au départ, avec quatre méthodes :
  - `ajouter(titre)` crée la tâche, l'ajoute à la fin de la liste et la renvoie ; un titre déjà présent lève une `ValueError` ;
  - `supprimer(tache)` la retire de la liste ;
  - `basculer(tache)` la marque faite si elle était à faire, et inversement ;
  - `filtrer(filtre)` renvoie la liste des tâches qui correspondent à l'un des `FILTRES`, dans l'ordre d'ajout ; un autre filtre lève une `ValueError`.

Les messages des `ValueError` s'adressent à l'utilisateur : l'interface les affichera tels quels.

La cellule commence par `%%writefile modele.py` : Jupyter écrit le reste de la cellule dans ce fichier au lieu de l'exécuter. La cellule de vérification, après la solution, teste votre modèle.

**Documentation utile**

- [%%writefile](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le cours sur les classes de données montre les deux outils dont `Tache` et `ListeTaches` ont besoin : `__post_init__()`, que la dataclass appelle après son `__init__` généré, valide le titre (`self.titre = self.titre.strip()`, puis `if not self.titre: ...`) ; `field(default_factory=list)` donne à chaque `ListeTaches` sa propre liste. `any(...)` sur un générateur dit si une tâche a déjà ce titre ; comparez au titre nettoyé, `"  Pain "` est un doublon de `"Pain"`. `basculer` n'a qu'une ligne : `tache.faite = not tache.faite`.

```python
%%writefile modele.py
from dataclasses import dataclass, field

FILTRES = ("toutes", "à faire", "faites")


@dataclass
class Tache:
  titre: str
  faite: bool = False

  def __post_init__(self) -> None:
    self.titre = self.titre.strip()
    if not self.titre:
      msg = "une tâche a besoin d'un titre"
      raise ValueError(msg)


@dataclass
class ListeTaches:
  taches: list[Tache] = field(default_factory=list)

  def ajouter(self, titre: str) -> Tache:
    tache = Tache(titre)
    if any(...):
      msg = f"« {tache.titre} » est déjà dans la liste"
      raise ValueError(msg)
    ...

  def supprimer(self, tache: Tache) -> None: ...

  def basculer(self, tache: Tache) -> None: ...

  def filtrer(self, filtre: str) -> list[Tache]:
    if filtre == "toutes":
      return list(self.taches)
    ...
```

### Solution

In [ ]:
%%writefile modele.py
from dataclasses import dataclass, field

FILTRES = ("toutes", "à faire", "faites")


@dataclass
class Tache:
  titre: str
  faite: bool = False

  def __post_init__(self) -> None:
    self.titre = self.titre.strip()
    if not self.titre:
      msg = "une tâche a besoin d'un titre"
      raise ValueError(msg)


@dataclass
class ListeTaches:
  taches: list[Tache] = field(default_factory=list)

  def ajouter(self, titre: str) -> Tache:
    tache = Tache(titre)
    if any(autre.titre == tache.titre for autre in self.taches):
      msg = f"« {tache.titre} » est déjà dans la liste"
      raise ValueError(msg)
    self.taches.append(tache)
    return tache

  def supprimer(self, tache: Tache) -> None:
    self.taches.remove(tache)

  def basculer(self, tache: Tache) -> None:
    tache.faite = not tache.faite

  def filtrer(self, filtre: str) -> list[Tache]:
    if filtre == "toutes":
      return list(self.taches)
    if filtre == "à faire":
      return [tache for tache in self.taches if not tache.faite]
    if filtre == "faites":
      return [tache for tache in self.taches if tache.faite]
    msg = f"filtre inconnu : {filtre!r}"
    raise ValueError(msg)

### Vérification

La cellule suivante recharge `modele.py` (le notebook garde sinon la version qu'il a importée en premier), puis joue un petit scénario. Résultats attendus : `['Acheter du pain']` à faire, `['Réviser le cours']` faite, puis trois `ValueError` (titre vide, doublon, filtre inconnu), et enfin `['Acheter du pain']` après la suppression.

In [ ]:
import importlib

import modele

importlib.reload(modele)

liste = modele.ListeTaches()
liste.ajouter("Acheter du pain")
revision = liste.ajouter("  Réviser le cours ")
liste.basculer(revision)
print("À faire :", [tache.titre for tache in liste.filtrer("à faire")])
print("Faites :", [tache.titre for tache in liste.filtrer("faites")])

for essai in (
  lambda: liste.ajouter("   "),
  lambda: liste.ajouter("Acheter du pain"),
  lambda: liste.filtrer("urgentes"),
):
  try:
    essai()
  except ValueError as erreur:
    print("ValueError :", erreur)
  else:
    print("Pas de ValueError : à revoir")

liste.supprimer(revision)
print("Toutes :", [tache.titre for tache in liste.filtrer("toutes")])

## 2. La vue

Écrivez `vue.py` : une classe `VueTaches` qui hérite de `ttk.Frame`, comme la `VueCompteur` du cours. Elle ne connaît pas le modèle : elle n'importe de `modele.py` que `FILTRES`. Elle crée ses widgets et les place avec `grid` :

| Ligne | Widgets |
|---|---|
| 0 | `saisie`, un `ttk.Entry` lié à `titre`, un `tk.StringVar` ; le bouton `ajouter` (« Ajouter ») |
| 1 | `liste`, une `tk.Listbox` où l'on peut sélectionner plusieurs tâches, sur les deux colonnes |
| 2 | sur les deux colonnes, un `ttk.Frame` qui contient, à gauche, les boutons `basculer` (« Faite / à faire ») et `supprimer` (« Supprimer ») et, à droite, `choix_filtre`, une `ttk.Combobox` des `FILTRES` liée à `filtre`, un `tk.StringVar` qui vaut `"toutes"` au départ |
| 3 | `etat`, un `ttk.Label` |

Quand la fenêtre s'agrandit, la saisie s'élargit et la liste grandit dans les deux sens. Le contrôleur se servira de ces noms d'attributs : gardez-les.

La vue a trois méthodes, que le contrôleur appellera :

- `afficher(lignes, etat)` remplace le contenu de la liste par `lignes`, une liste de chaînes, et le texte d'`etat` par `etat` ;
- `selection()` renvoie la liste des indices des lignes sélectionnées ;
- `signaler(message)` affiche le message dans une boîte d'erreur.

Les cellules qui suivent la solution ouvrent un aperçu de la vue, avec deux lignes d'exemple : les boutons n'y font encore rien.

**Documentation utile**

- [Listbox (TkDocs)](https://tkdocs.com/tutorial/morewidgets.html#listbox)
- [Combobox (TkDocs)](https://tkdocs.com/tutorial/widgets.html#combobox)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `tk.Listbox(self, height=10, selectmode="extended")` : Maj+clic et Ctrl+clic sélectionnent plusieurs lignes.
- `ttk.Combobox(actions, textvariable=self.filtre, values=FILTRES, state="readonly")` : `state="readonly"` interdit d'y taper un filtre qui n'existe pas.
- La liste s'étend sur deux colonnes avec `columnspan=2` et remplit sa cellule avec `sticky="nsew"` ; `self.columnconfigure(0, weight=1)` et `self.rowconfigure(1, weight=1)` donnent l'espace gagné à la colonne de la saisie et à la ligne de la liste.
- Dans le `ttk.Frame` de la ligne 2, `pack(side="left")` et `pack(side="right")` : un autre conteneur, donc un autre gestionnaire de placement que `grid`.
- `insert("end", *lignes)` insère toutes les lignes d'un coup ; `curselection()` renvoie un tuple d'indices.

```python
%%writefile vue.py
import tkinter as tk
from tkinter import messagebox, ttk

from modele import FILTRES


class VueTaches(ttk.Frame):
  def __init__(self, parent) -> None:
    super().__init__(parent, padding=10)
    self.titre = tk.StringVar()
    self.saisie = ttk.Entry(self, textvariable=self.titre)
    self.saisie.grid(row=0, column=0, sticky="ew")
    self.ajouter = ttk.Button(self, text="Ajouter")
    self.ajouter.grid(row=0, column=1)
    self.liste = ...
    actions = ttk.Frame(self)
    actions.grid(row=2, column=0, columnspan=2, sticky="ew")
    self.basculer = ttk.Button(actions, text="Faite / à faire")
    self.basculer.pack(side="left")
    ...
    self.etat = ttk.Label(self)
    self.etat.grid(row=3, column=0, columnspan=2, sticky="w")
    self.columnconfigure(0, weight=1)
    self.rowconfigure(1, weight=1)

  def afficher(self, lignes: list[str], etat: str) -> None:
    self.liste.delete(0, "end")
    ...

  def selection(self) -> list[int]:
    return ...

  def signaler(self, message: str) -> None:
    messagebox.showerror("Tâches", message)
```

### Solution

In [ ]:
%%writefile vue.py
import tkinter as tk
from tkinter import messagebox, ttk

from modele import FILTRES


class VueTaches(ttk.Frame):
  def __init__(self, parent) -> None:
    super().__init__(parent, padding=10)
    self.titre = tk.StringVar()
    self.saisie = ttk.Entry(self, textvariable=self.titre)
    self.saisie.grid(row=0, column=0, sticky="ew")
    self.ajouter = ttk.Button(self, text="Ajouter")
    self.ajouter.grid(row=0, column=1)
    self.liste = tk.Listbox(self, height=10, selectmode="extended")
    self.liste.grid(row=1, column=0, columnspan=2, sticky="nsew")
    actions = ttk.Frame(self)
    actions.grid(row=2, column=0, columnspan=2, sticky="ew")
    self.basculer = ttk.Button(actions, text="Faite / à faire")
    self.basculer.pack(side="left")
    self.supprimer = ttk.Button(actions, text="Supprimer")
    self.supprimer.pack(side="left")
    self.filtre = tk.StringVar(value=FILTRES[0])
    self.choix_filtre = ttk.Combobox(
      actions, textvariable=self.filtre, values=FILTRES, state="readonly", width=8
    )
    self.choix_filtre.pack(side="right")
    self.etat = ttk.Label(self)
    self.etat.grid(row=3, column=0, columnspan=2, sticky="w")
    self.columnconfigure(0, weight=1)
    self.rowconfigure(1, weight=1)
    for enfant in self.winfo_children():
      enfant.grid_configure(padx=5, pady=5)

  def afficher(self, lignes: list[str], etat: str) -> None:
    self.liste.delete(0, "end")
    self.liste.insert("end", *lignes)
    self.etat.configure(text=etat)

  def selection(self) -> list[int]:
    return list(self.liste.curselection())

  def signaler(self, message: str) -> None:
    messagebox.showerror("Tâches", message)

### Aperçu

La première cellule écrit un petit programme qui n'affiche que la vue, la seconde le lance. Agrandissez la fenêtre : la saisie et la liste doivent suivre.

In [ ]:
%%writefile apercu.py
import tkinter as tk

from vue import VueTaches

root = tk.Tk()
root.title("Aperçu de la vue")
vue = VueTaches(root)
vue.pack(fill="both", expand=True)
vue.afficher(["[ ] Acheter du pain", "[x] Réviser Tkinter"], "1 tâche(s) à faire")
root.mainloop()

In [ ]:
lancer("apercu.py")

## 3. Le contrôleur

Écrivez `app.py` : une classe `Controleur` qui reçoit le modèle et la vue, branche les widgets sur ses méthodes et tient la vue à jour, puis une fonction `main` qui lance l'application.

- `rafraichir()` demande au modèle les tâches du filtre choisi, les garde dans `self.visibles`, et les affiche, une ligne par tâche, sous la forme `[ ] titre` (à faire) ou `[x] titre` (faite), avec le nombre de tâches à faire dans `etat`. La ligne `i` de la liste est la tâche `self.visibles[i]`.
- `ajouter()` ajoute la tâche saisie. Si le modèle lève une `ValueError`, la vue signale son message ; sinon, la saisie se vide et la liste se rafraîchit. Le bouton « Ajouter » et la touche Entrée dans la saisie l'appellent.
- `basculer()` et `supprimer()` agissent sur les tâches sélectionnées, puis rafraîchissent la liste. Un double-clic sur une tâche la bascule.
- Choisir un filtre rafraîchit la liste.

`main()` crée la fenêtre (avec un titre et une taille minimale), y place la vue, qui occupe tout l'espace, crée le contrôleur avec une `ListeTaches` vide, puis lance la boucle d'événements. `app.py` l'appelle quand on l'exécute comme un script.

La cellule qui suit la solution lance l'application.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Un bouton reçoit la méthode sans l'appeler : `vue.ajouter.configure(command=self.ajouter)`.
- Un callback de `bind` reçoit l'événement, dont on n'a pas besoin ici : `vue.saisie.bind("<Return>", lambda event: self.ajouter())`. Le double-clic est `"<Double-Button-1>"`, le choix d'un filtre `"<<ComboboxSelected>>"`, sur `vue.choix_filtre`.
- `try` / `except ValueError as erreur` / `else`, comme dans le contrôleur du cours : le `else` ne s'exécute que si l'ajout a réussi.
- La liste n'affiche que les tâches du filtre courant : sa ligne `i` n'est pas `modele.taches[i]`, d'où `self.visibles`. Une méthode `selectionnees()` qui renvoie `[self.visibles[i] for i in self.vue.selection()]` sert à `basculer` et à `supprimer`.
- `if __name__ == "__main__":` en fin de fichier : `main()` ne s'exécute pas quand un autre module importe `app`.

```python
%%writefile app.py
import tkinter as tk

from modele import ListeTaches, Tache
from vue import VueTaches


class Controleur:
  def __init__(self, modele: ListeTaches, vue: VueTaches) -> None:
    self.modele = modele
    self.vue = vue
    self.visibles: list[Tache] = []
    vue.ajouter.configure(command=self.ajouter)
    vue.saisie.bind("<Return>", lambda event: self.ajouter())
    ...
    self.rafraichir()

  def rafraichir(self) -> None:
    self.visibles = self.modele.filtrer(self.vue.filtre.get())
    lignes = []
    for tache in self.visibles:
      ...
    restantes = ...
    self.vue.afficher(lignes, f"{restantes} tâche(s) à faire")

  def ajouter(self) -> None:
    try:
      ...
    except ValueError as erreur:
      ...
    else:
      ...

  def selectionnees(self) -> list[Tache]:
    return [self.visibles[index] for index in self.vue.selection()]

  def basculer(self) -> None: ...

  def supprimer(self) -> None: ...


def main() -> None:
  root = tk.Tk()
  ...
  root.mainloop()


if __name__ == "__main__":
  main()
```

### Solution

In [ ]:
%%writefile app.py
import tkinter as tk

from modele import ListeTaches, Tache
from vue import VueTaches


class Controleur:
  def __init__(self, modele: ListeTaches, vue: VueTaches) -> None:
    self.modele = modele
    self.vue = vue
    self.visibles: list[Tache] = []
    vue.ajouter.configure(command=self.ajouter)
    vue.saisie.bind("<Return>", lambda event: self.ajouter())
    vue.basculer.configure(command=self.basculer)
    vue.liste.bind("<Double-Button-1>", lambda event: self.basculer())
    vue.supprimer.configure(command=self.supprimer)
    vue.choix_filtre.bind("<<ComboboxSelected>>", lambda event: self.rafraichir())
    self.rafraichir()

  def rafraichir(self) -> None:
    self.visibles = self.modele.filtrer(self.vue.filtre.get())
    lignes = []
    for tache in self.visibles:
      case = "[x]" if tache.faite else "[ ]"
      lignes.append(f"{case} {tache.titre}")
    restantes = len(self.modele.filtrer("à faire"))
    self.vue.afficher(lignes, f"{restantes} tâche(s) à faire")

  def ajouter(self) -> None:
    try:
      self.modele.ajouter(self.vue.titre.get())
    except ValueError as erreur:
      self.vue.signaler(str(erreur))
    else:
      self.vue.titre.set("")
      self.rafraichir()

  def selectionnees(self) -> list[Tache]:
    return [self.visibles[index] for index in self.vue.selection()]

  def basculer(self) -> None:
    for tache in self.selectionnees():
      self.modele.basculer(tache)
    self.rafraichir()

  def supprimer(self) -> None:
    for tache in self.selectionnees():
      self.modele.supprimer(tache)
    self.rafraichir()


def main() -> None:
  root = tk.Tk()
  root.title("Tâches")
  root.minsize(320, 300)
  vue = VueTaches(root)
  vue.pack(fill="both", expand=True)
  Controleur(ListeTaches(), vue)
  root.mainloop()


if __name__ == "__main__":
  main()

### Lancement

Essayez : ajoutez des tâches avec le bouton et avec Entrée, puis un titre vide et un doublon ; basculez plusieurs tâches d'un coup (Ctrl+clic) et une autre d'un double-clic ; changez de filtre ; supprimez. Le TP jumeau en ipywidgets reprend `modele.py` sans y changer une ligne : seuls la vue et le contrôleur changent.

In [ ]:
lancer("app.py")